# Copia pública del 22 de octubre de 2026

Código de investigación; revisar entradas en `../README.md`. Las salidas guardadas se retiraron. Las etapas 01–06 no se reejecutaron para esta publicación. Para la síntesis numérica utilizada en la memoria, ejecutar `07_Sintesis_resultados_20261022.ipynb`. Las utilidades interactivas son opcionales de revisión y no una dependencia de la síntesis científica.

In [ ]:
from pathlib import Path
import sys
BASE = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'Configuracion.json').is_file()), None)
if BASE is None:
    raise FileNotFoundError('Iniciar Jupyter en notebooks/actual_20261022 o su subcarpeta de notebooks.')
sys.path.insert(0, str(BASE))
import publicacion_config


# 05 · Red completa de drenaje, ksn y perfiles χ alrededor de fallas
**19 de octubre de 2026.** Este notebook muestra la red extraída del DEM, no sólo los 18 ríos anteriores ni sólo los cauces que cruzan una falla. Incluye todas las ramas y todos los órdenes desde un umbral explícito de **0,1 km² de área aportante**, que puede cambiarse.

El mapa colorea cada segmento por ksn obtenido en una ventana de **1000 m a lo largo del cauce**. Se presentan el polígono completo y vistas ampliadas de Carranza, Pelluhue y El Yolki sobre hillshade. El visor HTML funciona sin instalar ni iniciar otro servidor: permite zoom, desplazamiento, consulta de cada segmento y cambio del área mínima visible.

No existe una red de drenaje DEM sin un umbral de inicio de canal: incluir todos los píxeles de escorrentía equivaldría a incluir también laderas. Aquí el umbral queda explícito y las cabeceras se conservan en el mapa. Los filtros para escoger perfiles χ del notebook anterior no se aplican a esta red.

## Abrir los resultados de esta corrida

[**Mapa de toda la red de ksn**](../README.md) · [**Perfiles χ con litología**](../README.md)

Las imágenes también están guardadas abajo. Al recalcular, utilizar los enlaces que imprimen las celdas de exportación de esa nueva corrida.

## 1. Parámetros editables
Cambiar `VENTANA_KSN_M` para la longitud de ventana y `AREA_MINIMA_KM2` para la densidad de la red. Después, reiniciar el kernel y ejecutar todas las celdas. Cada ejecución crea su carpeta; el grafo se reutiliza en caché si sus fuentes y parámetros no cambian.

El visor permite ocultar los cauces de área menor a un valor; para incluir cauces por debajo de 0,1 km² se debe reducir el parámetro y recalcular. No se inventan datos de ramas ausentes.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import sys,json,importlib
import numpy as np
import geopandas as gpd
from IPython.display import display,Image,HTML
BASE=Path(str(BASE / 'inputs_external' / 'Jupyter_Memoria'))
APOYO=BASE/'Red_drenaje_poligono_20261019'
if str(APOYO) not in sys.path:sys.path.insert(0,str(APOYO))
import red_fallas as rf
import mapa_red_completa as mr
rf=importlib.reload(rf);mr=importlib.reload(mr)
CONFIG=json.loads((BASE/'Configuracion.json').read_text(encoding='utf-8-sig'))
DEM=BASE/CONFIG['dem']
AREA=BASE/'Productos/Datos/v1/topografia/area_aportante_12m_v1.tif'
D8=BASE/'Productos/Datos/v1/topografia/flow_direction_D8_12m_v1.tif'
POLIGONO=APOYO/'Fuentes/PoligonoFallas.shp'
FALLAS=BASE/CONFIG['chaf']
AREA_MINIMA_KM2=0.10
VENTANA_KSN_M=1000.0
THETA=0.40
A0=1e6
MIN_NODOS_KSN=5
FRACCION_MINIMA_VENTANA=0.45
ELEVACION_MINIMA_DEM_M=1e-6
RUN='20261019_'+datetime.now(timezone.utc).strftime('%H%M%S_%fZ')
SALIDA=BASE/'Productos/Red_drenaje_poligono'/RUN
SALIDA.mkdir(parents=True,exist_ok=False)
print('Resultados:',SALIDA)

## 2. Extraer todas las ramas y conservar el aporte exterior
Se utilizan D8 y área aportante calculados para el DEM completo; no se calcula flujo sobre el polígono recortado. Primero se conserva la conectividad, se verifica área no decreciente aguas abajo y ausencia de ciclos; después se recortan las líneas al polígono.

Se conserva el filtro del raster para excluir el relleno costero con cota numéricamente casi nula (`Z≤10⁻⁶ m`). Este filtro no convierte el datum vertical y no selecciona exclusivamente cauces en roca. La cartografía regional puede incluir planicies y depósitos aluviales.

In [ ]:
poligonos,poligono=rf.leer_poligono(POLIGONO)
fallas=rf.leer_fallas(FALLAS,poligono)
grafo,control_grafo=rf.extraer_grafo(DEM,AREA,D8,AREA_MINIMA_KM2*1e6,APOYO/'cache',THETA,A0,ELEVACION_MINIMA_DEM_M)
print('Área del polígono:',round(poligono.area/1e6,2),'km²')
print('Fallas:',', '.join(fallas.nombre_base.unique()))

## 3. Calcular ksn para cada segmento de todas las ramas
Se integra χ trapezoidalmente con A₀=10⁶ m² y θ=0,40. Cada ventana de longitud de cauce 1000 m se ajusta mediante regresión lineal de z contra χ, con intercepto libre:
$$\chi=\int(A_0/A)^{\theta}ds,\qquad k_{sn}=\frac{dz}{d\chi}A_0^{\theta}.$$
En confluencias se sigue aguas arriba la rama de mayor área aportante; los tributarios usan contexto aguas abajo para no interrumpir su ventana. Todas las ramas se conservan en el mapa; pendientes no positivas o ajustes insuficientes se muestran en gris, sin asignarles un ksn positivo ficticio.

Las ventanas parciales requieren al menos cinco nodos y el 45% de la longitud solicitada; su longitud efectiva se exporta. La escala de colores común utiliza P5–P95, sin recortar los valores de los CSV o del GeoPackage. [Perron y Royden (2013)](https://dspace.mit.edu/entities/publication/515de692-d600-499d-b8a2-2a9084d858e9).

In [ ]:
ajustes,dentro,control_ksn=rf.calcular_ksn(grafo,poligono,VENTANA_KSN_M,MIN_NODOS_KSN,FRACCION_MINIMA_VENTANA)
red=rf.segmentos_aoi(grafo,poligono,dentro,ajustes)
red=red[red.geom_type.eq('LineString')].copy()
assert len(red)>0 and red.geometry.intersects(poligono).all()
assert red.area_km2.ge(AREA_MINIMA_KM2-1e-8).all()
print('Segmentos de TODOS los cauces en el polígono:',len(red))
print('Con ksn válido:',int(red.ksn.gt(0).sum()))
print('Segmentos de cabecera/orden 1 incluidos:',int(red.orden.eq(1).sum()))

## 4. Ver los colores de ksn alrededor de las fallas
El primer mapa incluye toda la red del polígono. Los siguientes mapas amplían cada falla, conservando la misma escala de color. Azul/violeta representa ksn menor; amarillo/rojo, ksn mayor. El ksn normaliza la pendiente por área de drenaje: no es por sí solo una medida de actividad de falla.

En el HTML: elegir la falla en **Vista**, usar la rueda para acercarse, arrastrar para moverse y pasar el cursor para consultar el valor de cada segmento. La opción **Resaltar ±1 km** es sólo una ayuda espacial alrededor de las trazas; no define una zona deformada medida.

In [ ]:
mapas_png,visor_html,control_mapa=mr.crear_mapas(red,fallas,poligono,DEM,SALIDA,VENTANA_KSN_M,AREA_MINIMA_KM2)
for file in mapas_png:
    display(Image(filename=str(file),width=1150))

## 5. Guardar para QGIS y abrir el visor desde Jupyter
Se conservan todos los segmentos, no sólo los asociados a perfiles seleccionados. GeoPackage: capas `red_completa_ksn`, `fallas` y `poligono`. El CSV contiene ksn, área, orden, χ, R², RMSE y longitud efectiva de ventana. La geometría permanece en el GeoPackage.

Una coincidencia espacial entre una traza y un cambio de ksn deberá contrastarse con confluencias, litología y resolución/ruido del DEM. Las ramas inferiores de esta red regional no heredan automáticamente los cortes de tus ríos de calibración.

In [ ]:
GIS=SALIDA/'Red_completa_ksn_PoligonoFallas.gpkg'
red.to_file(GIS,layer='red_completa_ksn',driver='GPKG')
fallas.to_file(GIS,layer='fallas',driver='GPKG')
poligonos.to_file(GIS,layer='poligono',driver='GPKG')
red.drop(columns='geometry').to_csv(SALIDA/'Todos_los_segmentos_ksn.csv.gz',index=False,compression='gzip')
sx=np.arange(0,5001,12.);test=rf.ventana_ols(sx,.2*sx,10+4*.2*sx,W=VENTANA_KSN_M)
assert np.allclose(test['ksn'][np.isfinite(test['ksn'])],4*A0**THETA,rtol=1e-8)
control={'parametros':{'ventana_m':VENTANA_KSN_M,'area_minima_km2':AREA_MINIMA_KM2,'theta':THETA,'A0':A0},
 'grafo':control_grafo,'ksn':control_ksn,'mapa':control_mapa,'sin_filtrar_por_orden_o_cruce_de_falla':True,
 'area_poligono_km2':poligono.area/1e6,'n_nodos_poligono':int(dentro.sum()),
 'sha256':{str(p):rf.sha(p) for p in [POLIGONO,*POLIGONO.parent.glob('PoligonoFallas.*'),FALLAS,APOYO/'red_fallas.py',APOYO/'mapa_red_completa.py']}}
(SALIDA/'Control_y_fuentes.json').write_text(json.dumps(control,ensure_ascii=False,indent=2),encoding='utf8')
(APOYO/'Ultima_corrida.json').write_text(json.dumps({'salida':str(SALIDA),'notebook':str(BASE/'Notebooks_principales_20261019/05_Red_completa_fallas_20261019.ipynb')},ensure_ascii=False,indent=2),encoding='utf8')
from urllib.parse import quote
from html import escape
href='/files/'+quote(visor_html.relative_to(Path(str(BASE))).as_posix(),safe='/')
display(HTML('<h3><a href="'+escape(href,quote=True)+'" target="_blank">ABRIR MAPA INTERACTIVO DE TODA LA RED</a></h3>'))
print('También puedes abrir este HTML con Edge o Chrome, sin servidor adicional:',visor_html)
print('Archivo QGIS:',GIS)

## 6. Perfiles χ de los cauces que cruzan las fallas, con litología
Esta sección recupera la parte de perfiles que antes estaba mezclada con τ. No calcula K ni τ.

La red del mapa conserva todas las cabeceras. Para los perfiles completos se exige orden ≥2 en el cruce y longitud ≥2 km, criterios editables. Los perfiles se siguen hasta su salida/cabecera D8 del DEM completo, sin recortarlos al polígono. Pueden compartir cauces inferiores; no son réplicas independientes.

La litología se asigna por intersección directa. Gris significa sin cobertura cartográfica; las superposiciones ambiguas no se rellenan por cercanía. La cobertura integrada es limitada en el norte. Esta sección se calcula cuando ejecutes el notebook; las figuras previas del mapa se conservaron de la corrida ya guardada.

In [ ]:
GENERAR_PERFILES_CHI=True
ORDEN_MINIMO_CRUCE=2
LONGITUD_MINIMA_CAUCE_M=2000.0
if GENERAR_PERFILES_CHI:
    import figuras_red as fr
    cruces=rf.cruzar_fallas(red,fallas)
    perfiles_falla,seleccion,control_seleccion=rf.seleccionar_perfiles(grafo,cruces,ORDEN_MINIMO_CRUCE,LONGITUD_MINIMA_CAUCE_M)
    perfiles_falla,unidades=rf.asignar_litologia(perfiles_falla,BASE/CONFIG['geologia_integrada'])
    cruces_perfiles=rf.cruces_en_perfiles(perfiles_falla,fallas)
    figuras_perfiles,html_perfiles,resumen_perfiles=fr.plot_perfiles(perfiles_falla,cruces_perfiles,SALIDA/'Perfiles_chi',VENTANA_KSN_M)
    control_seleccion.to_csv(SALIDA/'Control_seleccion_perfiles.csv',index=False,encoding='utf-8-sig')
    cruces_perfiles.to_csv(SALIDA/'Perfiles_chi/Cruces_falla_perfiles.csv',index=False,encoding='utf-8-sig')
    resumen_perfiles.to_csv(SALIDA/'Perfiles_chi/Resumen_perfiles.csv',index=False,encoding='utf-8-sig')
    for f in figuras_perfiles:display(Image(filename=str(f),width=1300))
    from urllib.parse import quote
    link='/files/'+quote(html_perfiles.relative_to(Path(str(BASE))).as_posix(),safe='/')
    display(HTML('<a target="_blank" href="'+link+'">Abrir perfiles χ con litología y fallas</a>'))

    from shapely.geometry import LineString,Point
    lineas=gpd.GeoDataFrame([{'perfil_id':pid,'longitud_m':float(p.distancia_m.iloc[-1]),
        'fraccion_sin_lito':float(p.codigo_lito.eq('Sin cobertura').mean()),
        'geometry':LineString(p[['x_utm','y_utm']].to_numpy())} for pid,p in perfiles_falla.items()],crs='EPSG:32718')
    if len(lineas):lineas.to_file(GIS,layer='perfiles_completos',driver='GPKG')
    if len(cruces_perfiles):
        gcr=gpd.GeoDataFrame(cruces_perfiles.copy(),geometry=[Point(x,y) for x,y in zip(cruces_perfiles.x_cruce,cruces_perfiles.y_cruce)],crs='EPSG:32718')
        gcr.to_file(GIS,layer='cruces_en_perfiles_completos',driver='GPKG')
    control['perfiles_chi']={'n_cruces_red':len(cruces),'n_perfiles':len(perfiles_falla),
        'orden_minimo':ORDEN_MINIMO_CRUCE,'longitud_minima_m':LONGITUD_MINIMA_CAUCE_M,
        'geologia_sha256':rf.sha(BASE/CONFIG['geologia_integrada'])}
    (SALIDA/'Control_y_fuentes.json').write_text(json.dumps(control,ensure_ascii=False,indent=2),encoding='utf8')
